[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Authentication


## What you will be able to do

Send an API key or an access token where an API expects it, read both from the environment instead
of writing them in your code, trade a client id and secret for a token and renew it when it expires,
tell a `401` from a `403`, and keep every credential out of your output and your repository.


## The idea

### The problem

Every request so far was answered for anyone who asked. Many APIs answer only callers they know,
because they hold each caller to a limit, charge for use, or serve data that is not public. The
practice API has an endpoint like that, `/me`, which says who sent a request, and a request that
does not say gets `401 Unauthorized`.

A server remembers nothing of the requests that came before, so every request carries its own proof
of who sent it: a **credential**, usually a long string. Whoever holds the string can call the API as
you, use up your limits and add to your bill, which makes a credential a password for a program. It
also ends up in more places than a password does: in the code that sends it, in the output of a
cell that prints a request to see what went wrong, in a server's log when it travels in a URL, and in
a repository, committed with the code or the notebook that holds it.

### What authentication is

> **Authentication** is how a request proves who sent it, with a credential that the server checks
> before it answers. An **API key** is a credential issued once and sent unchanged with every
> request. An **access token** is a credential that expires, which a program often gets by sending
> its **client id** and **client secret** to the API's token endpoint. Both usually travel in the
> `Authorization` header, as the word `Bearer` and the credential. **Authorization** is what a
> caller may do once the server knows who it is, often granted as **scopes**.

### Why it works that way

- **Every request carries the credential.** HTTP keeps nothing between requests, so the server
  checks a credential every time, and a client sends one every time.
- **A header, not the query.** A server logs the first line of every request, and that line holds
  the path and the query. A key in the query is written into the log, a browser's history and error
  messages, and a key in a header is not.
- **Bearer means whoever holds it.** A server accepts a bearer credential from anyone who sends it,
  and asks for no other proof. HTTPS keeps it private on the way, by encrypting the whole request,
  headers included.
- **`401` is about who, and `403` about what.** `401` says the server does not know who is asking,
  and its `WWW-Authenticate` header says what to send. `403` says the server knows, and the answer is
  no. HTTP's names mix the two up: the `Authorization` header carries authentication, and
  `401 Unauthorized` means unauthenticated.
- **A token expires, so a leaked one stops working.** A leaked key works until someone revokes it.
  A leaked token works only until it expires, often within an hour, and the secret that gets new
  tokens is sent only to the token endpoint.
- **The environment keeps credentials out of code.** A program reads its credentials from
  environment variables, so the code can be shared, printed and committed, and whoever runs it
  supplies their own.

### Where this shows up

OpenAI's API and Anthropic's take a key in `Authorization: Bearer`, and Anthropic's also still
accepts its older header of its own, `x-api-key`. api.data.gov, which manages the APIs of NASA and
other US federal agencies, accepts a key in an `X-Api-Key` header or an `api_key` query parameter.
GitHub's API takes a personal access token in `Authorization: Bearer`. Spotify's Web API gives an
application a token that lasts an hour, for a client id and secret sent with Basic authentication,
as this notebook does. Colab keeps keys in its Secrets panel, and the **Hosting an API** notebook
gives a secret to a server running on a host.

### What this notebook covers

- A request with no credential, `401 Unauthorized`, and what `WWW-Authenticate` asks for
- A key read from the environment and checked without printing it, and where to set one
- A `.env` file that git ignores, and what to do about a key that was committed
- A key in the `Authorization` header, in a header of the API's own, and in the query, and the log
  line that keeps the query
- `403 Forbidden`, for a credential without the scope a request needs
- An access token for a client id and secret, what Basic authentication sends, and what a token
  holds
- A token that has expired
- A client that fetches its token, sends it, and gets a new one when it expires
- Five errors, from an environment variable that is not set to a dataclass that shows a secret

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import os
import requests

for headers in [{}, {"Authorization": f"Bearer {os.environ['PRACTICE_API_KEY']}"}]:
    response = requests.get("http://127.0.0.1:8765/me", headers=headers, timeout=10)
    print(response.status_code, response.json())
```

```
401 {'error': 'this endpoint needs an API key or an access token'}
200 {'client': 'station-report', 'scopes': ['stations:read'], 'credential': 'API key'}
```

One request without a credential and one with, and the key read from an environment variable, so
that it appears nowhere in the code.


## Setup

Thirteen imports, the last of them the practice API.

- `requests` sends every request, and its `auth=` sends a client id and secret
- `os` reads credentials from the environment, where this cell puts the practice API's
- `base64` decodes what Basic authentication sends, and the parts of a token
- `json` reads the claims inside a token
- `datetime` and `timezone` turn the times in a token into dates
- `time` tells a client when its token is due to be renewed
- `tempfile` makes folders for a small project and a key file
- `dataclass` and `field` build a class that holds credentials, in Common errors
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here, and writes files
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`.
  `credentials()` returns its credentials, named as environment variables

The practice API's credentials are made up, and open nothing else. Putting them in the environment
stands in for a step that happens before a program of your own starts, which the worked examples
describe, so no cell in this notebook contains a credential.


In [1]:
import base64
import importlib
import json
import os
import sys
import tempfile
import time
import urllib.request
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
os.environ.update(practice_api.credentials())    # where a program finds its credentials
print("The practice API is running at", BASE)
print("Credentials in the environment:", ", ".join(practice_api.credentials()))


The practice API is running at http://127.0.0.1:8765
Credentials in the environment: PRACTICE_API_KEY, PRACTICE_CLIENT_ID, PRACTICE_CLIENT_SECRET


## Worked examples

### A request with no credential

`/me` responds with who sent a request, which makes it the place to see what the practice API makes
of a credential. First, a request with none, and a request for a station to compare it with:


In [2]:
response = requests.get(f"{BASE}/me", timeout=10)

print(response.status_code, response.reason, response.json())
print("WWW-Authenticate:", response.headers["WWW-Authenticate"])
print("a station:", requests.get(f"{BASE}/stations/tromso", timeout=10).status_code)


401 Unauthorized {'error': 'this endpoint needs an API key or an access token'}
WWW-Authenticate: Bearer realm="practice-api"
a station: 200


An API decides for each endpoint whether it needs a credential, and the practice API's stations are
open to anyone. `WWW-Authenticate` is the header a `401` must carry. It names the **scheme** the
server expects, here `Bearer`, and a `realm`, a name for the part of the server the credential is
for.

### A key from the environment

A program reads its credentials from **environment variables**, the named text values a process
starts with. Python keeps them in `os.environ`, which works like a dictionary. Setup put the practice
API's credentials there:


In [3]:
KEY = os.environ["PRACTICE_API_KEY"]

print("PRACTICE_API_KEY:", len(KEY), "characters, ending", KEY[-4:])
print("PRACTICE_TOKEN is set:", "PRACTICE_TOKEN" in os.environ)
print(os.environ.get("PRACTICE_TOKEN"))


PRACTICE_API_KEY: 36 characters, ending 9c71
PRACTICE_TOKEN is set: False
None


The first line confirms which key was loaded without showing it: a length and the last four
characters are enough to tell two keys apart, and not enough to use either. Looking up a name that
is not set with square brackets raises `KeyError`, which Common errors shows, and `get` returns
`None` instead.

Environment variables are set before a program starts, and where to set one depends on where the
program runs:

| Where the code runs | Where the key is set | How the code reads it |
|---|---|---|
| a terminal on macOS or Linux | `export PRACTICE_API_KEY=...`, then start Jupyter or Python | `os.environ["PRACTICE_API_KEY"]` |
| PowerShell on Windows | `$env:PRACTICE_API_KEY = "..."`, then start Jupyter or Python | `os.environ["PRACTICE_API_KEY"]` |
| a project on your computer | a line `PRACTICE_API_KEY=...` in a file named `.env` | `load_dotenv()` from the python-dotenv package, then `os.environ` |
| Colab | the Secrets panel, under the key icon in the left sidebar | `userdata.get("PRACTICE_API_KEY")`, with `userdata` from `google.colab` |

A process copies its environment when it starts, so a variable set in a terminal after Jupyter
started is not in the notebook's environment. A command typed in a terminal is also saved in the
shell's history, key and all, which is one reason projects keep keys in a `.env` file. Colab's
secrets are not in `os.environ`: `userdata.get` reads them, once a secret's notebook access is
switched on.

### A .env file that git ignores

A `.env` file holds a project's credentials in plain text, one `NAME=value` to a line, so it must
never be committed. Here is a small project that git tracks, with its code in `report.py` and its
key in `.env`:


In [4]:
project = Path(tempfile.mkdtemp()) / "station-report"
project.mkdir()
(project / "report.py").write_text('import os\n\nKEY = os.environ["PRACTICE_API_KEY"]\n')
(project / ".env").write_text(f"PRACTICE_API_KEY={KEY}\n")

!git -C "{project}" init --quiet
!git -C "{project}" status --short


?? .env
?? report.py


`??` marks a file that git sees and does not track yet, and `git add .` would add both files, the
key with them. A line in a file named `.gitignore` tells git to act as if a file were not there:


In [5]:
(project / ".gitignore").write_text(".env\n")

!git -C "{project}" status --short
!git -C "{project}" check-ignore --verbose .env


?? .gitignore
?? report.py
.gitignore:1:.env	.env


`.env` is no longer listed, and `check-ignore` names the line of `.gitignore` that hides it. Commit
`.gitignore` in a project's first commit, before any `.env` exists. It protects only the files it
names: `git add --force` still adds an ignored file, and a key pasted into `report.py` itself is
committed with the code.

A notebook is committed with its outputs, so a key printed in a cell is committed too. That is why
no cell in this notebook prints a credential, even a made-up one: the cells print facts about it
instead.

If a key does reach a repository, revoke it at the service that issued it, and make a new one. A
commit that deletes the key does not help, because every earlier commit still holds it, and so does
every copy of the repository made in the meantime. GitHub scans public repositories for keys in the
formats of the services it works with, and tells those services, so that they can revoke them.

### A key in a header, or in the query

With the key loaded, it goes where the API's documentation says. Many APIs take `Bearer`, a space
and the key, in the `Authorization` header. Some name a header of their own, and some take the key
as a query parameter. The practice API accepts all three:


In [6]:
in_authorization = requests.get(f"{BASE}/me", headers={"Authorization": f"Bearer {KEY}"}, timeout=10)
in_own_header = requests.get(f"{BASE}/me", headers={"X-API-Key": KEY}, timeout=10)
in_query = requests.get(f"{BASE}/me", params={"api_key": KEY}, timeout=10)

for label, response in [("Authorization", in_authorization), ("X-API-Key", in_own_header), ("api_key", in_query)]:
    print(f"{label:<14} {response.status_code} {response.json()}")
print("sent:", list(in_authorization.request.headers))


Authorization  200 {'client': 'station-report', 'scopes': ['stations:read'], 'credential': 'API key'}
X-API-Key      200 {'client': 'station-report', 'scopes': ['stations:read'], 'credential': 'API key'}
api_key        200 {'client': 'station-report', 'scopes': ['stations:read'], 'credential': 'API key'}
sent: ['User-Agent', 'Accept-Encoding', 'Accept', 'Connection', 'Authorization']


All three reached the practice API, which knows the key as `station-report`, and `scopes` lists what
the key may do. The last line prints the names of the headers sent, as the **Headers and Content
Types** notebook did, which confirms that `Authorization` went without showing its value.
`print(in_authorization.request.headers)` would show the key.

### Where a key in the query ends up

A server writes a line to its **access log** for every request it answers. The practice API keeps
those lines instead of printing them, and `practice_api.access_log()` returns them. Here are the
lines for the three requests, with the key replaced by the name of its variable:


In [7]:
lines = practice_api.access_log()[-3:]

for line in lines:
    print(line.replace(KEY, "<PRACTICE_API_KEY>"))
print("lines that hold the key:", sum(KEY in line for line in lines))


127.0.0.1 - - [01/Mar/2026 09:00:00] "GET /me HTTP/1.1" 200 -
127.0.0.1 - - [01/Mar/2026 09:00:00] "GET /me HTTP/1.1" 200 -
127.0.0.1 - - [01/Mar/2026 09:00:00] "GET /me?api_key=<PRACTICE_API_KEY> HTTP/1.1" 200 -
lines that hold the key: 1


A line records the caller's address, the time, the request's first line and the status code. The
first line of a request holds its path and its query, and none of its headers, so the key sent in a
header left no trace, and the key sent in the query is in the log for anyone who reads it. The same
URL is written into the logs of other servers along the way, such as a load balancer's, and into a
browser's history. requests puts it in the message of the error that `raise_for_status` raises, as
it does for this request, which the key is not allowed to make:


In [8]:
try:
    requests.get(f"{BASE}/network/maintenance", params={"api_key": KEY}, timeout=10).raise_for_status()
except requests.HTTPError as error:
    print(str(error).replace(KEY, "<PRACTICE_API_KEY>"))


403 Client Error: Forbidden for url: http://127.0.0.1:8765/network/maintenance?api_key=<PRACTICE_API_KEY>


Left uncaught, that message is printed in a traceback, and a notebook saves its tracebacks with its
outputs. So send a key in a header wherever an API allows it.

### 403: a key without the scope

The key was refused at `/network/maintenance` in the query, and it is refused in a header too:


In [9]:
response = requests.get(f"{BASE}/network/maintenance", headers={"Authorization": f"Bearer {KEY}"}, timeout=10)

print(response.status_code, response.reason, response.json())
print("WWW-Authenticate:", response.headers["WWW-Authenticate"])


403 Forbidden {'error': 'this needs the maintenance:read scope, and this API key has only stations:read'}
WWW-Authenticate: Bearer realm="practice-api", error="insufficient_scope", scope="maintenance:read"


A **scope** names something a credential may do. The maintenance schedule needs `maintenance:read`,
and the key was issued with `stations:read` alone, so the practice API knew who was asking and
answered `403 Forbidden`, with the scope it needs in `WWW-Authenticate`. The **Status Codes** notebook
drew the line between the two codes: `401` is about who is asking, and `403` about what they may do.
Sending this key again, in any place, changes nothing. The fix is a credential with the scope, and
the practice API issues one to its maintenance console, in a different way.

### An access token for a client id and secret

Some APIs issue a key once. Others register an application with a **client id** and a **client
secret**, which the application trades for an access token that expires. That trade is OAuth 2.0's
**client credentials grant**: a `POST` to the token endpoint, with the id and secret sent in Basic
authentication and `grant_type=client_credentials` sent as a form field. The **Sending Data** notebook
covers `POST` and the bodies it carries; here it is only the way a token endpoint is asked.


In [10]:
CLIENT_ID = os.environ["PRACTICE_CLIENT_ID"]
CLIENT_SECRET = os.environ["PRACTICE_CLIENT_SECRET"]

response = requests.post(f"{BASE}/auth/token", auth=(CLIENT_ID, CLIENT_SECRET),
                         data={"grant_type": "client_credentials"}, timeout=10)
grant = response.json()
token = grant["access_token"]

print(response.status_code, "Cache-Control:", response.headers["Cache-Control"])
print({name: value for name, value in grant.items() if name != "access_token"})
print("access_token:", len(token), "characters")


200 Cache-Control: no-store
{'token_type': 'Bearer', 'expires_in': 3600, 'scope': 'stations:read maintenance:read'}
access_token: 220 characters


`auth=` sent the id and secret, and `data=` sent the form field. The response holds the token and
what a client needs to know about it: `token_type` says the token is sent as `Bearer`, as the key
was, `expires_in` says it lasts 3,600 seconds, an hour, and `scope` lists what it may do,
`maintenance:read` included. `Cache-Control: no-store` tells any cache along the way not to keep a
copy. The practice API accepts the id and secret only in Basic authentication, which OAuth 2.0
requires a server to support for a client with a secret; some servers also take them as form
fields.

Here is what `auth=` put in the request:


In [11]:
scheme, encoded = response.request.headers["Authorization"].split(" ")
decoded = base64.b64decode(encoded).decode("utf-8")

print(scheme)
print("the id and the secret, joined by a colon:", decoded == f"{CLIENT_ID}:{CLIENT_SECRET}")
print("the part before the colon:", decoded.partition(":")[0])


Basic
the id and the secret, joined by a colon: True
the part before the colon: maintenance-console


Basic authentication is the id, a colon and the secret, written in **Base64**, an encoding that
spells any bytes with letters, digits and a few symbols. An encoding is not encryption: anyone who
sees the header gets the secret back in one line, as that cell did. What keeps it private on the way
is HTTPS. The practice API uses `http` because its requests never leave the machine they are made
on. A real API's address starts with `https`, and a client should send credentials to no other kind.

### The token in the Authorization header

The token goes where the key went:


In [12]:
authorized = {"Authorization": f"Bearer {token}"}

print(requests.get(f"{BASE}/me", headers=authorized, timeout=10).json())

response = requests.get(f"{BASE}/network/maintenance", headers=authorized, timeout=10)
print(response.status_code, response.reason)
for visit in response.json()["visits"]:
    print(f"  {visit['date']}  {visit['station']:<9} {', '.join(visit['work'])}")


{'client': 'maintenance-console', 'scopes': ['stations:read', 'maintenance:read'], 'credential': 'access token'}
200 OK
  2026-03-09  svalbard  clear the rain gauge of snow, calibrate the rain gauge
  2026-03-16  oslo      calibrate the rain gauge, calibrate the anemometer
  2026-03-23  tromso    calibrate the anemometer


The practice API knew the token as the maintenance console's, with the scope the schedule needs. The
schedule follows from the gaps in `/network` that the **JSON in a Response** notebook read:
Svalbard's rain gauge is buried in snow, two rain gauges were never calibrated, and two anemometers
were last calibrated more than a year before these visits.

### What a token holds

OAuth 2.0 leaves what is inside a token to the server, and a client sends a token as it received
it. Many services, the practice API among them, issue **JSON Web Tokens**, or JWTs: three parts
separated by dots, each written in Base64. The middle part holds the token's **claims**:


In [13]:
header, payload, signature = token.split(".")
claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))

print(claims)
print("issued: ", datetime.fromtimestamp(claims["iat"], tz=timezone.utc))
print("expires:", datetime.fromtimestamp(claims["exp"], tz=timezone.utc))


{'sub': 'maintenance-console', 'scope': 'stations:read maintenance:read', 'iat': 1772355600, 'exp': 1772359200}
issued:  2026-03-01 09:00:00+00:00
expires: 2026-03-01 10:00:00+00:00


`sub` is the subject, the client the token speaks for, and `iat` and `exp` are when it was issued
and when it expires, in seconds since the start of 1970, UTC. A JWT's Base64 uses `-` and `_` in
place of `+` and `/`, and leaves off the `=` padding at the end, so the cell adds the padding back
and decodes with `urlsafe_b64decode`. The practice API's clock stops at the moment in its `Date`
header, so its tokens are issued at 09:00 UTC on March 1, 2026, on every run.

A client still takes a token's lifetime from `expires_in`, not from `exp`, because another service's
token may not be a JWT at all. And anyone who holds a token can read its claims, so decode a real
token in your own code, as here, and never paste one into a website. What stops a holder from
changing a claim is the signature, which the server computed from the other two parts with a key
only the server has:


In [14]:
wider = dict(claims, scope=claims["scope"] + " admin")
changed = base64.urlsafe_b64encode(json.dumps(wider).encode("utf-8")).decode("ascii").rstrip("=")
response = requests.get(f"{BASE}/me", headers={"Authorization": f"Bearer {header}.{changed}.{signature}"}, timeout=10)

print(response.status_code, response.json())
print("WWW-Authenticate:", response.headers["WWW-Authenticate"])


401 {'error': 'the key or access token is not valid'}
WWW-Authenticate: Bearer realm="practice-api", error="invalid_token"


The changed claims asked for `admin`, the signature no longer matched them, and the practice API
refused the token. Its signing key is in `practice_api.py` for anyone to read, which is fine for
practice and never true of a real service.

### A token that has expired

`/auth/expired-token` hands out a token for the maintenance console that expired a day before the
practice API's clock, which is how any token looks once its hour is up:


In [15]:
expired = requests.get(f"{BASE}/auth/expired-token", timeout=10).json()["access_token"]
response = requests.get(f"{BASE}/network/maintenance", headers={"Authorization": f"Bearer {expired}"}, timeout=10)

print(response.status_code, response.json())
print("WWW-Authenticate:", response.headers["WWW-Authenticate"])


401 {'error': 'the access token has expired'}
WWW-Authenticate: Bearer realm="practice-api", error="invalid_token", error_description="the access token has expired"


`error="invalid_token"` is how OAuth 2.0 says that a token has expired, been revoked or been
damaged, and that a client may get a new token and try again. That is the difference from the `403`:
a new token fixes this `401`, and no token fixes a missing scope. A careful client renews a token in
two ways: shortly before `expires_in` runs out, and whenever a `401` says `invalid_token`, since a
token can also be revoked early.

### A client that fetches, sends and renews its token

What a program needs from this notebook, in one small client. `StationClient` gets its credentials
from whoever creates it, and never shows the secret. It fetches a token the first time it needs one
and sends it with every request. It fetches a new token a minute before `expires_in` runs out, and
again when the practice API answers `401` with `invalid_token`:


In [16]:
class StationClient:
    """Requests to the practice API with an access token, fetched when needed and renewed when refused."""

    def __init__(self, base, client_id, client_secret):
        self.base = base
        self.client_id = client_id
        self._secret = client_secret
        self._token = None
        self._renew_at = 0.0                             # a time.monotonic() reading
        self.tokens_fetched = 0

    def __repr__(self):
        return f"StationClient({self.base!r}, client_id={self.client_id!r})"

    def _fetch_token(self):
        response = requests.post(f"{self.base}/auth/token", auth=(self.client_id, self._secret),
                                 data={"grant_type": "client_credentials"}, timeout=10)
        response.raise_for_status()
        grant = response.json()
        self._token = grant["access_token"]
        self._renew_at = time.monotonic() + grant["expires_in"] - 60
        self.tokens_fetched += 1

    def _send(self, path):
        return requests.get(f"{self.base}{path}", headers={"Authorization": f"Bearer {self._token}"}, timeout=10)

    def get(self, path):
        """The JSON at path, sent with a token that is fetched or renewed as needed."""
        if self._token is None or time.monotonic() >= self._renew_at:
            self._fetch_token()
        response = self._send(path)
        if response.status_code == 401 and "invalid_token" in response.headers.get("WWW-Authenticate", ""):
            self._fetch_token()
            response = self._send(path)
        response.raise_for_status()
        return response.json()


client = StationClient(BASE, os.environ["PRACTICE_CLIENT_ID"], os.environ["PRACTICE_CLIENT_SECRET"])
print(client)
print(client.get("/me")["client"], "| tokens fetched:", client.tokens_fetched)
print(len(client.get("/network/maintenance")["visits"]), "visits | tokens fetched:", client.tokens_fetched)

client._token = expired                                  # the token as it will look once its hour is up
print(client.get("/me")["client"], "| tokens fetched:", client.tokens_fetched)


StationClient('http://127.0.0.1:8765', client_id='maintenance-console')
maintenance-console | tokens fetched: 1
3 visits | tokens fetched: 1
maintenance-console | tokens fetched: 2


### Where each part came from

| In the client | What it relies on | The section that showed it |
|---|---|---|
| `client_id` and `client_secret`, passed in | credentials read from the environment, never written in code | A key from the environment |
| `__repr__`, without the secret | a credential that is never printed | A .env file that git ignores |
| `auth=` and `data={"grant_type": "client_credentials"}` | a token for a client id and secret | An access token for a client id and secret |
| `"Authorization": f"Bearer {self._token}"` | a token sent in a header, not the query | The token in the Authorization header |
| `expires_in`, less a minute | a lifetime taken from the response, not from inside the token | What a token holds |
| a second try after `invalid_token` | a `401` that a new token fixes | A token that has expired |

The first two requests shared one token. The last request was sent with the expired token, planted
as a test would plant it, and the client fetched a second token and sent the request again. A `403`
is not tried again: `raise_for_status` raises it, because no new token adds a scope.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/09-authentication-solutions.ipynb).

**1.** Send `/me` the key in an `X-API-Key` header, and print the `client` and the `scopes` from the
response.


In [17]:
# your code here


**2.** Send `/network/maintenance` the key, and print the status code and the scope that the
`WWW-Authenticate` header asks for, taken out of the header's text.


In [18]:
# your code here


**3.** Get an access token, and print how many minutes it lasts and its scopes as a list, without
printing the token.


In [19]:
# your code here


**4.** Decode the claims of the token from `/auth/expired-token`, and print its `sub` and the moment
it expired, as a date and time in UTC.


In [20]:
# your code here


**5.** Write `redact(text)`, which returns `text` with the key and the client secret replaced by
`<PRACTICE_API_KEY>` and `<PRACTICE_CLIENT_SECRET>`. Send `/me` the key in the query, and print the
redacted URL of the request and the redacted last line of the access log.


In [21]:
# your code here


**6.** Write `get_maintenance(token)`, which returns the maintenance visits and the token it used,
getting a new token when the practice API answers `401` with `invalid_token`. Call it with the token
from `/auth/expired-token`, and print the number of visits and whether the token it used is a new
one.


In [22]:
# your code here


## Common errors

### KeyError: 'PRACTICE_KEY'


In [23]:
key = os.environ["PRACTICE_KEY"]


KeyError: 'PRACTICE_KEY'

No environment variable has that name. Three causes account for most of these: the name is spelled
differently from the one that was set, as here; the variable was set in a terminal after Jupyter
started, and a process copies its environment only when it starts; or the key is in Colab's Secrets
panel, which `userdata.get` reads and `os.environ` does not. To see which names are set, print the
names, never the values:


In [24]:
print(sorted(name for name in os.environ if name.startswith("PRACTICE_")))


['PRACTICE_API_KEY', 'PRACTICE_CLIENT_ID', 'PRACTICE_CLIENT_SECRET']


### HTTPError: 401 Client Error: Unauthorized for url: http://127.0.0.1:8765/me


In [25]:
response = requests.get(f"{BASE}/me", headers={"Authorization": KEY}, timeout=10)
response.raise_for_status()


HTTPError: 401 Client Error: Unauthorized for url: http://127.0.0.1:8765/me

The header held the key and nothing else. An `Authorization` value starts with a scheme, the word
that says what kind of credential follows, and a server that finds no scheme it uses treats the
request as if it carried no credential. The body says what it wanted:


In [26]:
print(response.json()["error"])

response = requests.get(f"{BASE}/me", headers={"Authorization": f"Bearer {KEY}"}, timeout=10)
print(response.status_code, response.json()["client"])


the Authorization header must be Bearer, a space, and a key or token
200 station-report


### ValueError: Invalid header value b'Bearer …\n'


In [27]:
key_file = Path(tempfile.mkdtemp()) / "key.txt"
key_file.write_text(KEY + "\n")                   # as echo and many editors save it, with a line break
key = key_file.read_text()

try:
    requests.get(f"{BASE}/me", headers={"Authorization": f"Bearer {key}"}, timeout=10)
except ValueError as error:
    print("ValueError:", str(error).replace(KEY, "…"))


ValueError: Invalid header value b'Bearer …\n'


`read_text` returns the whole file, the line break at its end included, and a header value cannot
hold a line break, because a line break is what ends a header. The cell caught the error and
printed its message with the key replaced, because the message quotes the whole value: left
uncaught, it would have printed the key into the notebook. Strip what you read from a file:


In [28]:
key = key_file.read_text().strip()
response = requests.get(f"{BASE}/me", headers={"Authorization": f"Bearer {key}"}, timeout=10)

print(response.status_code, response.json()["client"])


200 station-report


### HTTPError: 400 Client Error: Bad Request for url: http://127.0.0.1:8765/auth/token


In [29]:
response = requests.post(f"{BASE}/auth/token", auth=(CLIENT_ID, CLIENT_SECRET),
                         json={"grant_type": "client_credentials"}, timeout=10)
response.raise_for_status()


HTTPError: 400 Client Error: Bad Request for url: http://127.0.0.1:8765/auth/token

`json=` sent the grant type as a JSON body. OAuth 2.0 defines a token request as a form, so the
token endpoint found no `grant_type` field, and its body says so, in OAuth's `error` and
`error_description`:


In [30]:
print(response.json())

response = requests.post(f"{BASE}/auth/token", auth=(CLIENT_ID, CLIENT_SECRET),
                         data={"grant_type": "client_credentials"}, timeout=10)
print(response.status_code, response.json()["token_type"])


{'error': 'invalid_request', 'error_description': 'the request has no grant_type form field'}
200 Bearer


### No error, and the secret in the output: a dataclass that shows every field


In [31]:
@dataclass
class Credentials:
    client_id: str
    client_secret: str


credentials = Credentials(CLIENT_ID, CLIENT_SECRET)
print("print(credentials) would show the secret:", CLIENT_SECRET in repr(credentials))


print(credentials) would show the secret: True


A dataclass writes a `__repr__` that shows every field, as the **Dataclasses** notebook in the
**Object-Oriented Python** guide showed, and `print`, a cell's displayed value and debuggers all use
it. Nothing failed, so nothing draws attention to the secret on the screen or in the saved notebook.
The cell checked what printing would show, instead of printing it. `field(repr=False)`, which that
notebook used to keep a note out of a station's printout, leaves a field out:


In [32]:
@dataclass
class Credentials:
    client_id: str
    client_secret: str = field(repr=False)


print(Credentials(CLIENT_ID, CLIENT_SECRET))


Credentials(client_id='maintenance-console')


Pydantic, from the **Schemas and Validation** notebook, has a type for this, `SecretStr`, which
displays as asterisks and gives the value only to `get_secret_value()`.


## Recap

- A credential goes with every request. `401` means the server does not know who is asking, and
  `WWW-Authenticate` says what to send; `403` means it knows, and only a credential with the scope
  changes the answer.
- Read credentials from environment variables, and check one by its length, never by printing it.
- Send a key or a token in `Authorization: Bearer`, or in the header an API names. A key in the
  query is written into access logs and error messages.
- `auth=(client_id, client_secret)` with `data={"grant_type": "client_credentials"}` gets an access
  token. Basic authentication is an encoding, so credentials go only to `https` addresses.
- Anyone who holds a token can read its claims, and a changed claim breaks its signature. Renew a
  token before `expires_in` runs out, and when a `401` says `invalid_token`.
- Keep `.env` in `.gitignore`. A committed key stays in the history, so revoke it and make a new
  one.


## What is next

The **Pagination** notebook. Every response here arrived in one piece. That notebook reads a
collection that an API sends a page at a time, with `page` and `per_page` parameters and with
cursors, in a loop that stops when the pages run out.


---

&#8592; **Previous:** [Headers and Content Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/08-headers-and-content-types.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [Pagination](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/10-pagination.ipynb) &#8594;
